# Chapter 2: Overview of Text Classification
**Module 03 - Deep Learning for Text with PyTorch**  
*Source integrated from `chapter2.pdf`*

This notebook turns text into labels. It expands the original notebook with every major concept and code path from the PDF: classification types, embeddings, CNN classifiers, RNN/LSTM/GRU variants, and evaluation metrics.


## Learning Objectives

By the end of this notebook, you will be able to:

- Distinguish binary, multi-class, and multi-label text classification.
- Explain why embeddings improve on sparse encodings.
- Build text classifiers with embeddings, CNNs, RNNs, LSTMs, and GRUs.
- Evaluate classification models with accuracy, precision, recall, and F1.


## 2.1 Text Classification Defined

Text classification assigns labels to text, giving structure to unstructured language data.

| Classification type | Description | PDF example |
|---|---|---|
| Binary | Sort text into two categories | Email: `spam` vs `not spam` |
| Multi-class | Pick one label from multiple categories | News: politics, sports, technology |
| Multi-label | Assign multiple labels to one item | Books: action, adventure, fantasy |

Common applications include review sentiment analysis, spam detection, and tagging news articles with relevant topics.


## 2.2 Word Embeddings

Earlier encodings are useful first steps, but they often create too many sparse features and cannot identify similar words. Word embeddings map words to dense numerical vectors, allowing a model to learn relationships such as:

- `king` and `queen`
- `man` and `woman`

The standard neural pipeline is:

```text
Text -> Tokenization -> Word-to-index mapping -> Embedding layer -> Classifier
```


In [ ]:
# PDF snippet: word-to-index mapping and nn.Embedding
import torch
from torch import nn

words = ["The", "cat", "sat", "on", "the", "mat"]
word_to_idx = {word: i for i, word in enumerate(words)}
inputs = torch.LongTensor([word_to_idx[w] for w in words])

embedding = nn.Embedding(num_embeddings=len(words), embedding_dim=10)
output = embedding(inputs)

print("Word to index:", word_to_idx)
print("Input indices:", inputs.tolist())
print("Embedding output shape:", output.shape)
print(output)


## 2.3 Using Embeddings in a Pipeline

The PDF shows embeddings as a drop-in replacement after token-to-index mapping. A `Dataset` stores encoded token IDs; the embedding layer converts those IDs into dense vectors during model execution.


In [ ]:
# PDF-inspired embedding pipeline
from torch.utils.data import Dataset, DataLoader


def simple_preprocess_sentences(text):
    tokens = text.lower().replace(".", "").split()
    vocabulary = {"<PAD>": 0, "<UNK>": 1}
    for token in tokens:
        if token not in vocabulary:
            vocabulary[token] = len(vocabulary)
    return [vocabulary.get(token, vocabulary["<UNK>"]) for token in tokens], vocabulary


class TextDataset(Dataset):
    def __init__(self, encoded_sentences):
        self.data = encoded_sentences

    def __len__(self):
        return len(self.data)

    def __getitem__(self, index):
        return self.data[index]


text = "Your sample text here."
encoded_tokens, embedding_vocab = simple_preprocess_sentences(text)
dataset = TextDataset(torch.tensor(encoded_tokens, dtype=torch.long))
dataloader = DataLoader(dataset, batch_size=2, shuffle=True)
embedding_layer = nn.Embedding(num_embeddings=len(embedding_vocab), embedding_dim=50)

for batch in dataloader:
    embedded_batch = embedding_layer(batch)
    print("Batch:", batch.tolist(), "->", embedded_batch.shape)


## 2.4 A Simple Embedding Classifier

The original notebook used mean pooling over token embeddings. Mean pooling creates one fixed-size sentence vector by averaging token embeddings across the sequence.


In [ ]:
class MeanPoolingTextClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, num_classes):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.fc1 = nn.Linear(embed_dim, 64)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.3)
        self.fc2 = nn.Linear(64, num_classes)

    def forward(self, x):
        embedded = self.embedding(x)
        pooled = embedded.mean(dim=1)
        out = self.fc1(pooled)
        out = self.relu(out)
        out = self.dropout(out)
        return self.fc2(out)


toy_batch = torch.tensor([[1, 2, 3, 4], [1, 5, 0, 0]])
toy_model = MeanPoolingTextClassifier(vocab_size=8, embed_dim=10, num_classes=2)
print("Logits shape:", toy_model(toy_batch).shape)


## 2.5 Padding Variable-Length Sequences

Texts have different lengths. Padding lets us batch them into one rectangular tensor.


In [ ]:
from torch.nn.utils.rnn import pad_sequence

seq1 = torch.tensor([1, 2, 3, 4, 5])
seq2 = torch.tensor([1, 2])
seq3 = torch.tensor([1, 2, 3])

padded = pad_sequence([seq1, seq2, seq3], batch_first=True, padding_value=0)
print("Padded batch:")
print(padded)
print("Shape:", padded.shape)


## 2.6 CNNs for Text Classification

CNNs can classify text by sliding filters across token embeddings. The PDF frames this as tweet or sentiment classification.

| CNN component | Role for text |
|---|---|
| Filter/kernel | Slides across neighboring token embeddings. |
| Stride | Controls how far the filter moves at each step. |
| Convolutional layer | Learns local n-gram-like patterns. |
| Pooling layer | Reduces sequence length while preserving important signals. |
| Fully connected layer | Converts learned features into class logits. |


In [ ]:
# PDF snippets: CNN model definition and forward pass
import torch.nn.functional as F
import torch.optim as optim


class SentimentAnalysisCNN(nn.Module):
    def __init__(self, vocab_size, embed_dim):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.conv = nn.Conv1d(
            in_channels=embed_dim,
            out_channels=embed_dim,
            kernel_size=3,
            stride=1,
            padding=1,
        )
        self.fc = nn.Linear(embed_dim, 2)

    def forward(self, text):
        embedded = self.embedding(text).permute(0, 2, 1)
        conved = F.relu(self.conv(embedded))
        conved = conved.mean(dim=2)
        return self.fc(conved)


In [ ]:
# PDF snippets: preparing, training, and running the CNN sentiment model
book_samples = [
    ("the story was captivating and kept me hooked until the end".split(), 1),
    ("i found the characters shallow and the plot predictable".split(), 0),
]

vocab = sorted({word for sentence, _ in book_samples for word in sentence})
word_to_idx = {word: i for i, word in enumerate(vocab)}
vocab_size = len(word_to_idx)
embed_dim = 10

model = SentimentAnalysisCNN(vocab_size, embed_dim)
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=0.1)

for epoch in range(10):
    for sentence, label in book_samples:
        model.zero_grad()
        sentence_tensor = torch.LongTensor([word_to_idx.get(w, 0) for w in sentence]).unsqueeze(0)
        outputs = model(sentence_tensor)
        label_tensor = torch.LongTensor([int(label)])
        loss = criterion(outputs, label_tensor)
        loss.backward()
        optimizer.step()

for sample, _ in book_samples:
    input_tensor = torch.tensor([word_to_idx[w] for w in sample], dtype=torch.long).unsqueeze(0)
    outputs = model(input_tensor)
    _, predicted_label = torch.max(outputs.data, 1)
    sentiment = "Positive" if predicted_label.item() == 1 else "Negative"
    print(f"Book Review: {' '.join(sample)}")
    print(f"Sentiment: {sentiment}\n")


## 2.7 RNNs for Text Classification

RNNs process text one word at a time and maintain a short-term memory. Compared with CNNs:

| Model | Best at |
|---|---|
| CNN | Spotting local patterns in chunks of text. |
| RNN | Preserving order and context across a sequence. |

Example from the PDF: `"I just love getting stuck in traffic."` may require context to detect sarcasm.


In [ ]:
# PDF recap: Dataset and DataLoader for sequence data
class SequenceTextDataset(Dataset):
    def __init__(self, text):
        self.text = text

    def __len__(self):
        return len(self.text)

    def __getitem__(self, idx):
        return self.text[idx]


In [ ]:
# PDF-inspired RNN classifier
class RNNClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_size, output_size):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.rnn = nn.RNN(embed_dim, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        embedded = self.embedding(x)
        _, hidden = self.rnn(embedded)
        return self.fc(hidden.squeeze(0))


sample_tweet = "this movie had a great plot and amazing acting"
tweet_vocab = {"<PAD>": 0, **{w: i + 1 for i, w in enumerate(sample_tweet.split())}}
sample_tweet_tensor = torch.tensor([[tweet_vocab[w] for w in sample_tweet.split()]])
rnn_model = RNNClassifier(len(tweet_vocab), embed_dim=8, hidden_size=16, output_size=2)
sentiment_prediction = rnn_model(sample_tweet_tensor)
print("RNN logits:", sentiment_prediction)


## 2.8 LSTM and GRU Variations

Standard RNNs can struggle when useful evidence is far apart. The PDF introduces two gated variants:

| Variant | Key idea | Example use |
|---|---|---|
| LSTM | Input, forget, and output gates preserve longer context. | Mixed sentiment in longer reviews. |
| GRU | A simpler gated recurrent unit. | Fast recognition of spam-like patterns. |


In [ ]:
# PDF snippet: LSTM model
class LSTMModel(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        _, (hidden, _) = self.lstm(x)
        output = self.fc(hidden.squeeze(0))
        return output


# PDF snippet: GRU model
class GRUModel(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super().__init__()
        self.gru = nn.GRU(input_size, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        _, hidden = self.gru(x)
        output = self.fc(hidden.squeeze(0))
        return output


sequence_features = torch.randn(2, 5, 8)
print("LSTM logits:", LSTMModel(8, 16, 2)(sequence_features).shape)
print("GRU logits: ", GRUModel(8, 16, 2)(sequence_features).shape)


## 2.9 Evaluation Metrics for Text Classification

Accuracy alone can mislead. In the PDF example, if 9,800 out of 10,000 reviews are positive, a model that always predicts positive reaches 98% accuracy while failing to identify negative reviews.

| Metric | Meaning | Useful when |
|---|---|---|
| Accuracy | Correct predictions / all predictions | Classes are balanced. |
| Precision | Correct positive predictions / predicted positives | False positives are costly. |
| Recall | Correct positive predictions / actual positives | False negatives are costly. |
| F1 | Harmonic mean of precision and recall | Classes are imbalanced. |


In [ ]:
# PDF snippets: accuracy, precision, recall, and F1
# Install if needed: pip install torchmetrics
actual = torch.tensor([0, 1, 1, 0, 1, 0])
predicted = torch.tensor([0, 0, 1, 0, 1, 1])

try:
    from torchmetrics import Accuracy, Precision, Recall, F1Score

    accuracy = Accuracy(task="binary")
    precision = Precision(task="binary")
    recall = Recall(task="binary")
    f1 = F1Score(task="binary")

    print("Accuracy: ", accuracy(predicted, actual).item())
    print("Precision:", precision(predicted, actual).item())
    print("Recall:   ", recall(predicted, actual).item())
    print("F1 Score: ", f1(predicted, actual).item())
except Exception:
    tp = int(((predicted == 1) & (actual == 1)).sum())
    tn = int(((predicted == 0) & (actual == 0)).sum())
    fp = int(((predicted == 1) & (actual == 0)).sum())
    fn = int(((predicted == 0) & (actual == 1)).sum())

    acc = (tp + tn) / len(actual)
    prec = tp / (tp + fp)
    rec = tp / (tp + fn)
    f1_score = 2 * prec * rec / (prec + rec)

    print("Accuracy: ", acc)
    print("Precision:", prec)
    print("Recall:   ", rec)
    print("F1 Score: ", f1_score)


## Chapter Summary

| Area | What you learned |
|---|---|
| Classification types | Binary, multi-class, and multi-label classification solve different labeling problems. |
| Embeddings | Dense vectors let models learn semantic relationships. |
| CNN classifiers | Convolutions learn local text patterns. |
| RNN classifiers | Recurrent models preserve order and context. |
| LSTM/GRU | Gated models improve sequence memory. |
| Metrics | Precision, recall, and F1 are essential when accuracy hides class imbalance. |
